# Chapter 2: Hybridization and Molecular Geometry

## 1. Introduction

Welcome to the study of hybridization and molecular geometry, where we translate flat, 2D chemical drawings into the 3D shapes that govern their biological activity. The central goal of this section is to learn how to predict an atom's local 3D geometry from its hybridization. This skill is fundamental in medicinal chemistry, as the precise shape of a drug molecule determines whether it can successfully bind to its protein target, much like a key fitting into a lock. Our learning journey will begin with the rules for determining hybridization, then explore the specific geometries of sp³, sp², and sp hybridized atoms, and finally apply this knowledge to understand how the shapes of real drug molecules like Ritalin and Acetaminophen are critical to their function.

The animation below demonstrates that molecules are not planar. As the molecule adopts a stable 3D conformation, its energy drops. This spontaneous transition to a 3D shape proves that molecular bonds must be angled.

In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
import py3Dmol
import numpy as np
from IPython.display import HTML, display
import html

def create_planar_ring_conformation(mol, ring_radius=1.5):
    """
    Create a planar ring conformation by arranging atoms in a circle.
    All atoms placed at z=0 (perfectly flat).
    """
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    
    conf = mol.GetConformer()
    num_atoms = mol.GetNumAtoms()
    
    # Find the ring atoms (non-hydrogen)
    ring_atoms = []
    for atom in mol.GetAtoms():
        if atom.GetSymbol() != 'H':
            ring_atoms.append(atom.GetIdx())
    
    num_ring_atoms = len(ring_atoms)
    
    # Place ring atoms in a circle (planar)
    for i, atom_idx in enumerate(ring_atoms):
        angle = 2 * np.pi * i / num_ring_atoms
        x = ring_radius * np.cos(angle)
        y = ring_radius * np.sin(angle)
        z = 0.0
        conf.SetAtomPosition(atom_idx, (x, y, z))
    
    # Place hydrogens near their parent carbons (also planar)
    for atom_idx in range(num_atoms):
        atom = mol.GetAtomWithIdx(atom_idx)
        if atom.GetSymbol() == 'H':
            neighbors = atom.GetNeighbors()
            if neighbors:
                carbon_idx = neighbors[0].GetIdx()
                carbon_pos = conf.GetAtomPosition(carbon_idx)
                
                angle = np.arctan2(carbon_pos.y, carbon_pos.x)
                offset_angle = np.random.uniform(-0.3, 0.3)
                
                h_distance = ring_radius + 0.6
                h_x = h_distance * np.cos(angle + offset_angle)
                h_y = h_distance * np.sin(angle + offset_angle)
                h_z = 0.0
                
                conf.SetAtomPosition(atom_idx, (h_x, h_y, h_z))
    
    return mol

def generate_minimization_trajectory(mol, max_iters=300, kick_frequency=15, 
                                    kick_strength=0.3, convergence_threshold=0.005, 
                                    convergence_window=20, target_energy=25.46,
                                    minimize_steps_per_iter=1):
    """
    Generate trajectory with slower, more gradual optimization.
    STARTS WITH PLANAR GEOMETRY.
    """
    mol = create_planar_ring_conformation(mol)
    
    prop = AllChem.MMFFGetMoleculeProperties(mol)
    ff = AllChem.MMFFGetMoleculeForceField(mol, prop)
    
    trajectory_xyz = ""
    energies = []
    
    initial_energy = ff.CalcEnergy()
    energies.append(initial_energy)
    trajectory_xyz += Chem.MolToXYZBlock(mol)
    
    conf = mol.GetConformer()
    
    for i in range(max_iters):
        current_kick_strength = kick_strength
        if len(energies) > 50:
            recent_change = max(energies[-30:]) - min(energies[-30:])
            if recent_change < 0.2 and energies[-1] > target_energy:
                current_kick_strength = kick_strength * 1.5
        
        if i % kick_frequency == 0 and i > 0:
            for atom_idx in range(mol.GetNumAtoms()):
                pos = conf.GetAtomPosition(atom_idx)
                new_pos = (
                    pos.x + np.random.uniform(-current_kick_strength, current_kick_strength),
                    pos.y + np.random.uniform(-current_kick_strength, current_kick_strength),
                    pos.z + np.random.uniform(-current_kick_strength, current_kick_strength)
                )
                conf.SetAtomPosition(atom_idx, new_pos)
        
        ff.Minimize(maxIts=minimize_steps_per_iter)
        
        current_energy = ff.CalcEnergy()
        energies.append(current_energy)
        trajectory_xyz += Chem.MolToXYZBlock(mol)
        
        if current_energy < target_energy:
            print(f"✓ Target reached at frame {i+1}")
            break
        
        if len(energies) > convergence_window:
            energy_window = energies[-convergence_window:]
            energy_change = max(energy_window) - min(energy_window)
            
            if energy_change < convergence_threshold and i > 100:
                print(f"⚠ Converged at frame {i+1}")
                break
    
    return trajectory_xyz, energies

def create_interactive_viewer(trajectory_xyz, energies, width=760, height=500, target_energy=None):
    """
    Create an interactive viewer.
    Width is set to 760.
    Height (of the 3D viewer part) is set to 500.
    """
    num_frames = len(energies)
    viewer_id = f"viewer_{np.random.randint(100000, 999999)}"
    
    min_energy = min(energies)
    max_energy = max(energies)
    energy_range = max_energy - min_energy
    y_min = min_energy - energy_range * 0.1
    y_max = max_energy + energy_range * 0.1
    
    # Chart height fixed at 400 within the HTML
    chart_height = 400
    
    html_content = f"""
    <div style="font-family: Arial, sans-serif; max-width: {width}px; margin: 0 auto;">
        <div id="{viewer_id}" style="width: {width}px; height: {height}px; border: 2px solid #333; margin-bottom: 15px; position: relative;"></div>
        
        <div style="background: #f5f5f5; padding: 20px; border-radius: 8px; margin-bottom: 15px;">
            <div style="display: flex; align-items: center; gap: 15px; margin-bottom: 15px;">
                <button id="playBtn_{viewer_id}" style="padding: 10px 20px; font-size: 16px; cursor: pointer; background: #4CAF50; color: white; border: none; border-radius: 5px;">
                    ▶ Play
                </button>
                <button id="pauseBtn_{viewer_id}" style="padding: 10px 20px; font-size: 16px; cursor: pointer; background: #ff9800; color: white; border: none; border-radius: 5px;">
                    ⏸ Pause
                </button>
                <button id="resetBtn_{viewer_id}" style="padding: 10px 20px; font-size: 16px; cursor: pointer; background: #2196F3; color: white; border: none; border-radius: 5px;">
                    ⏮ Reset
                </button>
                <span id="frameInfo_{viewer_id}" style="font-weight: bold; font-size: 16px; margin-left: 15px;">Frame: 0 / {num_frames - 1}</span>
            </div>
            
            <div style="margin-bottom: 15px;">
                <label style="font-weight: bold; display: block; margin-bottom: 5px;">Frame Slider:</label>
                <input type="range" id="frameSlider_{viewer_id}" min="0" max="{num_frames - 1}" value="0" 
                       style="width: 100%; height: 8px; cursor: pointer;">
            </div>
            
            <div style="display: flex; gap: 30px; font-size: 14px;">
                <div>
                    <strong>Current:</strong> <span id="energyDisplay_{viewer_id}" style="font-size: 18px; color: #2196F3;">{energies[0]:.2f}</span>
                </div>
                <div>
                    <strong>Init:</strong> {energies[0]:.2f}
                </div>
                <div>
                    <strong>Final:</strong> {energies[-1]:.2f}
                </div>
            </div>
        </div>
        
        <div style="background: white; padding: 10px; border: 2px solid #333; border-radius: 8px;">
            <canvas id="energyChart_{viewer_id}" width="{width - 40}" height="{chart_height}"></canvas>
        </div>
    </div>
    
    <script src="https://3Dmol.csb.pitt.edu/build/3Dmol-min.js"></script>
    <script>
    (function() {{
        let viewer = $3Dmol.createViewer("{viewer_id}", {{backgroundColor: 'white'}});
        
        let xyzData = `{trajectory_xyz}`;
        
        viewer.addModelsAsFrames(xyzData, 'xyz');
        viewer.setStyle({{}}, {{stick: {{radius: 0.15}}, sphere: {{scale: 0.3}}}});
        viewer.zoomTo();
        viewer.render();
        
        let energies = {energies};
        let currentFrame = 0;
        let isPlaying = false;
        let playInterval = null;
        let playSpeed = 100;
        
        let slider = document.getElementById('frameSlider_{viewer_id}');
        let frameInfo = document.getElementById('frameInfo_{viewer_id}');
        let energyDisplay = document.getElementById('energyDisplay_{viewer_id}');
        let playBtn = document.getElementById('playBtn_{viewer_id}');
        let pauseBtn = document.getElementById('pauseBtn_{viewer_id}');
        let resetBtn = document.getElementById('resetBtn_{viewer_id}');
        
        // Energy chart setup
        let canvas = document.getElementById('energyChart_{viewer_id}');
        let ctx = canvas.getContext('2d');
        let chartWidth = canvas.width;
        let chartHeight = canvas.height;
        
        // INCREASED LEFT PADDING TO PREVENT Y-AXIS OVERLAP
        let padding = {{left: 85, right: 30, top: 30, bottom: 50}};
        
        let plotWidth = chartWidth - padding.left - padding.right;
        let plotHeight = chartHeight - padding.top - padding.bottom;
        
        let minEnergy = {y_min};
        let maxEnergy = {y_max};
        let targetEnergy = {target_energy if target_energy else 'null'};
        
        function drawEnergyChart(upToFrame) {{
            ctx.clearRect(0, 0, chartWidth, chartHeight);
            
            // Draw background
            ctx.fillStyle = '#f9f9f9';
            ctx.fillRect(padding.left, padding.top, plotWidth, plotHeight);
            
            // Draw grid
            ctx.strokeStyle = '#e0e0e0';
            ctx.lineWidth = 1;
            for (let i = 0; i <= 5; i++) {{
                let y = padding.top + (plotHeight / 5) * i;
                ctx.beginPath();
                ctx.moveTo(padding.left, y);
                ctx.lineTo(padding.left + plotWidth, y);
                ctx.stroke();
            }}
            
            // Draw axes
            ctx.strokeStyle = '#333';
            ctx.lineWidth = 2;
            ctx.beginPath();
            ctx.moveTo(padding.left, padding.top);
            ctx.lineTo(padding.left, padding.top + plotHeight);
            ctx.lineTo(padding.left + plotWidth, padding.top + plotHeight);
            ctx.stroke();
            
            // Draw target energy line if provided
            if (targetEnergy !== null) {{
                let targetY = padding.top + plotHeight - ((targetEnergy - minEnergy) / (maxEnergy - minEnergy)) * plotHeight;
                ctx.strokeStyle = '#ff9800';
                ctx.lineWidth = 2;
                ctx.setLineDash([5, 5]);
                ctx.beginPath();
                ctx.moveTo(padding.left, targetY);
                ctx.lineTo(padding.left + plotWidth, targetY);
                ctx.stroke();
                ctx.setLineDash([]);
                
                // Target label
                ctx.fillStyle = '#ff9800';
                ctx.font = 'bold 12px Arial';
                ctx.textAlign = 'right';
                ctx.fillText(`Target: ${{targetEnergy.toFixed(2)}}`, padding.left - 5, targetY + 4);
            }}
            
            // Draw energy curve up to current frame
            if (upToFrame > 0) {{
                ctx.strokeStyle = '#2196F3';
                ctx.lineWidth = 2.5;
                ctx.beginPath();
                
                for (let i = 0; i <= upToFrame; i++) {{
                    let x = padding.left + (i / ({num_frames - 1})) * plotWidth;
                    let y = padding.top + plotHeight - ((energies[i] - minEnergy) / (maxEnergy - minEnergy)) * plotHeight;
                    
                    if (i === 0) {{
                        ctx.moveTo(x, y);
                    }} else {{
                        ctx.lineTo(x, y);
                    }}
                }}
                ctx.stroke();
                
                // Draw current point
                let currentX = padding.left + (upToFrame / ({num_frames - 1})) * plotWidth;
                let currentY = padding.top + plotHeight - ((energies[upToFrame] - minEnergy) / (maxEnergy - minEnergy)) * plotHeight;
                
                ctx.fillStyle = '#f44336';
                ctx.beginPath();
                ctx.arc(currentX, currentY, 5, 0, 2 * Math.PI);
                ctx.fill();
                
                // Draw start point (green)
                let startX = padding.left;
                let startY = padding.top + plotHeight - ((energies[0] - minEnergy) / (maxEnergy - minEnergy)) * plotHeight;
                ctx.fillStyle = '#4CAF50';
                ctx.beginPath();
                ctx.arc(startX, startY, 6, 0, 2 * Math.PI);
                ctx.fill();
            }}
            
            // Y-axis labels
            ctx.fillStyle = '#333';
            ctx.font = '12px Arial';
            ctx.textAlign = 'right';
            for (let i = 0; i <= 5; i++) {{
                let energy = minEnergy + (maxEnergy - minEnergy) * (1 - i / 5);
                let y = padding.top + (plotHeight / 5) * i;
                ctx.fillText(energy.toFixed(1), padding.left - 10, y + 4);
            }}
            
            // X-axis labels
            ctx.textAlign = 'center';
            for (let i = 0; i <= 4; i++) {{
                let frame = Math.floor(({num_frames - 1}) * i / 4);
                let x = padding.left + (plotWidth * i / 4);
                ctx.fillText(frame, x, padding.top + plotHeight + 20);
            }}
            
            // Axis titles
            ctx.font = 'bold 14px Arial';
            ctx.fillStyle = '#333';
            
            // Y-axis title (Adjusted coordinates to fix overlap)
            ctx.save();
            // Moved X translate from 15 to 25 to give more room
            ctx.translate(25, chartHeight / 2);
            ctx.rotate(-Math.PI / 2);
            ctx.textAlign = 'center';
            ctx.fillText('Energy (kcal/mol)', 0, 0);
            ctx.restore();
            
            // X-axis title
            ctx.textAlign = 'center';
            ctx.fillText('Frame', padding.left + plotWidth / 2, chartHeight - 10);
            
            // Chart title
            ctx.font = 'bold 16px Arial';
            ctx.fillText('Energy Minimization Profile', chartWidth / 2, 20);
        }}
        
        function updateDisplay() {{
            frameInfo.textContent = `Frame: ${{currentFrame}} / {num_frames - 1}`;
            energyDisplay.textContent = energies[currentFrame].toFixed(2);
            slider.value = currentFrame;
            viewer.setFrame(currentFrame);
            viewer.render();
            drawEnergyChart(currentFrame);
        }}
        
        slider.addEventListener('input', function() {{
            currentFrame = parseInt(this.value);
            updateDisplay();
            if (isPlaying) {{
                stopPlayback();
            }}
        }});
        
        playBtn.addEventListener('click', function() {{
            if (!isPlaying) {{
                startPlayback();
            }}
        }});
        
        pauseBtn.addEventListener('click', function() {{
            stopPlayback();
        }});
        
        resetBtn.addEventListener('click', function() {{
            stopPlayback();
            currentFrame = 0;
            updateDisplay();
        }});
        
        function startPlayback() {{
            isPlaying = true;
            playBtn.style.background = '#888';
            playBtn.disabled = true;
            
            playInterval = setInterval(function() {{
                currentFrame++;
                if (currentFrame >= {num_frames - 1}) {{
                    currentFrame = {num_frames - 1};
                    stopPlayback();
                }}
                updateDisplay();
            }}, playSpeed);
        }}
        
        function stopPlayback() {{
            isPlaying = false;
            playBtn.style.background = '#4CAF50';
            playBtn.disabled = false;
            if (playInterval) {{
                clearInterval(playInterval);
                playInterval = null;
            }}
        }}
        
        updateDisplay();
    }})();
    </script>
    """
    
    # Iframe height must be set to 1100 exactly
    iframe_total_height = 1200
    
    # We use html.escape to safely put the content into srcdoc
    escaped_html = html.escape(html_content)
    
    iframe = f"""
    <iframe 
        width="100%" 
        height="{iframe_total_height}" 
        srcdoc="{escaped_html}" 
        frameborder="0" 
        allowfullscreen
    ></iframe>
    """
    
    return HTML(iframe)

def main():
    # --- USER SETTINGS ---
    target_smiles = "C1CCCCCCCCC1" 
    mol_name = "Cyclodecane"
    
    MINIMIZE_STEPS_PER_ITER = 1
    MAX_ITERATIONS = 250
    KICK_FREQUENCY = 20
    KICK_STRENGTH = 0.1
    CONVERGENCE_THRESHOLD = 0.005
    CONVERGENCE_WINDOW = 20
    TARGET_ENERGY = 25.00
    
    mol = Chem.MolFromSmiles(target_smiles)
    
    print(f"\n{mol_name} | {rdMolDescriptors.CalcMolFormula(mol)} | Target: {TARGET_ENERGY} kcal/mol\n")
    
    traj, energies = generate_minimization_trajectory(
        mol, 
        max_iters=MAX_ITERATIONS,
        kick_frequency=KICK_FREQUENCY,
        kick_strength=KICK_STRENGTH,
        convergence_threshold=CONVERGENCE_THRESHOLD,
        convergence_window=CONVERGENCE_WINDOW,
        target_energy=TARGET_ENERGY,
        minimize_steps_per_iter=MINIMIZE_STEPS_PER_ITER
    )
    
    # Pass specific height/width parameters here if needed, but defaults are now set.
    viewer = create_interactive_viewer(traj, energies, target_energy=TARGET_ENERGY)
    display(viewer)
    
    return viewer

if __name__ == "__main__":
    viewer = main()


Cyclodecane | C10H20 | Target: 25.0 kcal/mol



---

## 2. Key Concepts and Definitions

* **Hybridization**: The mixing of atomic orbitals to form new hybrid orbitals with specific geometries, enabling atoms to form bonds with defined spatial arrangements.
* **Molecular Geometry**: The three-dimensional arrangement of atoms in a molecule, determined by hybridization state and electron pair repulsion.
* **sp³ Hybridization**: One s and three p orbitals mix to form four equivalent hybrid orbitals arranged tetrahedrally (~109.5° bond angles).
* **sp² Hybridization**: One s and two p orbitals mix to form three equivalent hybrid orbitals arranged trigonally (~120° bond angles), with one unhybridized p orbital for π bonding.
* **sp Hybridization**: One s and one p orbital mix to form two equivalent hybrid orbitals arranged linearly (180° bond angles), with two unhybridized p orbitals for π bonding.
* **Lone Pairs**: Non-bonding electron pairs that occupy hybrid orbitals and affect molecular geometry.

---

## 3. Main Content

### 3.1 Atomic Orbitals

Every atoms have **atomic orbitals**. Formally, atomic orbitals are mathematical functions describing electron probability distributions, each with distinct shapes and energies that determine bonding capability. 

In plain English, atomic orbitals are region of space that contain electrons. The common orbitals important in our work would be the **s orbitals** and **p orbitals** 

**s Orbitals**

* Spherically symmetric
* Can hold 2 electrons

**p Orbitals**

* Dumbbell-shaped with two lobes
* Each can hold 2 electrons
* Three types of p orbitals (px, py and pz)
* Each of these orbitals are **orthogonal** (meaning 90°) to each other 

**d Orbitals**

* Complex shapes (four-lobed or doughnut)
* Five types of d orbitals (dxy, dxz, dyz, dx²-y², dz²)
* **RARELY** involved in organic medicinal chemistry

![Shapes of atomic orbitals](images/2_4_atomic_orbitals.png)
**Figure adapted from [source](https://scienceready.com.au/pages/schrodingers-atomic-model-spdf-notation?srsltid=AfmBOoqE4e7ZSR8uyXnZMa3OKRo1eaw5sTLS8cZfCrqXKjzqfzCCkKO2)**

These atomic orbitals can be **mixed** to form **new molecular orbitals** with different shapes. 

We call this mixing process **hybrdization**

For example, an **s orbital** (atomic orbital) can hybridize (mix) with a **p orbital** (atomic orbital) to form a **NEW sp orbital** (molecular orbital)

### 3.2 Hybridization and Molecular Orbitals

The ability of atoms like carbon, oxygen and nitrogen to adopt different hybridization states creates structural diversity in drug molecules. Each hybridization state produces distinct geometries that affect how molecules fit into binding sites.

#### 3.2.1 sp³ Hybridization (Tetrahedral)

An atom with four electron domains adopts sp³ hybridization, leading to a tetrahedral arrangement.

*   **Electron Domains:** 4
*   **Electron Geometry:** Tetrahedral
*   **Molecular Geometry:** Tetrahedral (with 0 lone pairs)
*   **Ideal Bond Angle:** ~109.5°

#### 3.2.2 sp² Hybridization (Trigonal Planar)

An atom with three electron domains is sp² hybridized, resulting in a flat, trigonal planar geometry.

*   **Electron Domains:** 3
*   **Electron Geometry:** Trigonal Planar
*   **Molecular Geometry:** Trigonal Planar (with 0 lone pairs)
*   **Ideal Bond Angle:** ~120°

#### 3.2.3 sp Hybridization (Linear)

An atom with two electron domains is sp hybridized, yielding a linear geometry.

*   **Electron Domains:** 2
*   **Electron Geometry:** Linear
*   **Molecular Geometry:** Linear (with 0 lone pairs)
*   **Ideal Bond Angle:** 180°


![Hybrization of orbitals](images/2_4_hybridization.png)
**Figure adapted from [source](https://stock.adobe.com/tr/images/sp3-sp2-sp-hybrid-orbitals/128894654)**

By looking at the type of bonds, you would be able to tease out what is the hybridization state of the atom, and from there deduce the likely shape based on the bond angles.

![Hybridization state of atoms in the molecule](images/2_4_molecule_hybridization_example.png)
**Figure adapted from [source](https://www.chemistrysteps.com/how-to-quickly-determine-sp3-sp2-and-sp-hybridization/)**

#### Distortion of Bond Angles like Lone Pair

### 3.4 The Impact of Lone Pairs on Heteroatoms

The presence of lone pairs on heteroatoms (like N and O) causes the molecular geometry (atom arrangement) to differ from the electron geometry (domain arrangement).

| Hybridization | Electron Domains | Lone Pairs | Electron Geometry | Molecular Geometry | Approx. Bond Angle |
| :--- | :--- | :--- | :--- | :--- | :--- |
| sp³ | 4 | 0 | Tetrahedral | Tetrahedral | ~109.5° |
| sp³ | 4 | 1 | Tetrahedral | Trigonal Pyramidal | ~107° |
| sp³ | 4 | 2 | Tetrahedral | Bent | ~104.5° |

> **Important:**
> Lone pairs are more repulsive than bonding pairs. This electron-electron repulsion "pushes" the bonding pairs closer together, compressing the bond angles from the ideal (e.g., 109.5° in a perfect tetrahedron) to smaller values (~107° in ammonia, ~104.5° in water). This distortion directly impacts how a molecule fits into a binding site.

*   **sp³ Nitrogen:** An amine nitrogen has four electron domains (3 bonds, 1 lone pair). The lone pair's repulsion compresses bond angles to ~107°, creating a **trigonal pyramidal** shape. This positions the lone pair to act as a potent hydrogen bond acceptor, as seen in Ritalin.
*   **sp³ Oxygen:** An alcohol oxygen has four electron domains (2 bonds, 2 lone pairs). The two lone pairs compress the bond angle to ~104.5°, resulting in a **bent** geometry, which precisely directs its lone pairs for binding, as in Acetaminophen's hydroxyl group.
*   **sp² Oxygen:** A carbonyl oxygen has three electron domains (1 double bond, 2 lone pairs), which adopt a trigonal planar electron geometry with ~120° angles, consistent with sp² hybridization. This geometry is critical for directing hydrogen bonds, as seen in Acetaminophen's carbonyl.

![Effect of lone pairs on bond angles](images/2_4_effect_of_lone_pairs_bond_angles.png)
**Figure adapted from [source](http://chemhume.co.uk/ASCHEM/Unit%201/Ch3Chemstr/Chemical%20Struct.htm)**

### 3.5 Special Case: The Planar Amide Nitrogen

Unlike a typical sp³ pyramidal amine, an amide nitrogen is a key exception. Due to resonance with the adjacent carbonyl, its lone pair delocalizes, forcing a flat, sp² trigonal planar geometry.

> **Important:**
> The planarity of the amide bond is a fundamental principle in protein structure. This rigidity, caused by resonance, prevents free rotation and forms the backbone of peptides and proteins, dictating how they fold into complex 3D structures like alpha-helices and beta-sheets. Understanding this exception is crucial for understanding drug-protein interactions.

Notice in the animation below that N-methylacetamide with an amide group (-NHCO) is planar, while dimethylamine with an amine group (-NH) group is not. 

In [2]:
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from IPython.display import display, HTML

# --- Helper: Molecule Generation ---
def get_mol_geometry(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3() 
    params.randomSeed = 0xf00d 
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    return mol, Chem.MolToMolBlock(mol)

# --- Helper: Base Viewer Setup ---
def init_comparison_viewer(width=760, height=760):
    view = py3Dmol.view(width=width, height=height, viewergrid=(1,2), linked=False)
    return view

def style_standard(view, mol_block, coordinates, title):
    view.addModel(mol_block, 'mol', viewer=coordinates)
    view.setStyle({'stick': {'radius': 0.2, 'colorscheme': 'darkCarbon'}}, viewer=coordinates)
    view.addLabel(title, {'position': {'x':0, 'y':-4, 'z':0}, 'backgroundColor': '#eee', 'fontColor': 'black'}, viewer=coordinates)
    view.zoomTo(viewer=coordinates)

def visualize_amide_vs_amine():
    view = init_comparison_viewer()
    
    # 1. Amide: N-methylacetamide (PLANAR around C-N bond)
    mol_amide, block_amide = get_mol_geometry('CC(=O)NC')
    style_standard(view, block_amide, (0,0), "Planar Amide")
    
    # 2. Amine: Dimethylamine (PYRAMIDAL at N)
    mol_amine, block_amine = get_mol_geometry('CNC')
    style_standard(view, block_amine, (0,1), "Pyramidal Amine")
    
    return view

view1 = visualize_amide_vs_amine()
view1.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## 4. Summary and Key Takeaways

In this section, we've explored how the concept of orbital hybridization allows us to predict the three-dimensional geometry of molecules, a critical skill for understanding drug-target interactions. We have seen that the shape of a molecule is rarely accidental; it is a direct consequence of the electronic structure of its atoms.

*   The number of electron domains (bonds and lone pairs) around an atom determines its hybridization: 4 domains = sp³, 3 domains = sp², and 2 domains = sp.
*   Hybridization dictates the electron geometry (tetrahedral, trigonal planar, linear), while the presence of lone pairs determines the final molecular geometry (e.g., trigonal pyramidal, bent).
*   Lone pairs are sterically demanding, compressing bond angles and serving as crucial hydrogen bond acceptors in biological systems.
*   Special cases like amides demonstrate that resonance can enforce planarity, overriding simple hybridization rules and creating rigid structural motifs essential for drug activity.